# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/suyashverma0/flyrank-ml-workspace2/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [3]:
import os

print("Current directory:", os.getcwd())

print("\nFiles in current directory:")
print(os.listdir("/content"))

print("\nSearching for dataset...")

matches = []

for root, dirs, files in os.walk("/content"):
    if "content_refresh_anonymized.csv" in files:
        matches.append(os.path.join(root, "content_refresh_anonymized.csv"))

print

Current directory: /content

Files in current directory:
['.config', 'sample_data']

Searching for dataset...


<function print(*args, sep=' ', end='\n', file=None, flush=False)>

In [5]:
import os

REPO_URL = "https://github.com/suyashverma0/flyrank-ml-workspace2.git"
REPO_PATH = "/content/flyrank-ml-workspace2"

if not os.path.exists(REPO_PATH):
    !git clone {REPO_URL} {REPO_PATH}
else:
    print("Repository already exists!")

print("Repository ready!")
print(os.listdir(REPO_PATH))

Cloning into '/content/flyrank-ml-workspace2'...
remote: Enumerating objects: 124, done.
remote: Counting objects: 100% (124/124), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 124 (delta 36), reused 97 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (124/124), 1.85 MiB | 6.60 MiB/s, done.
Resolving deltas: 100% (36/36), done.
Repository ready!
['DATA_USE.md', 'notebooks', '.gitignore', '.git', 'data', 'work', 'scripts', 'README.md', 'AGENTS.md', 'LICENSE', 'submission', 'docs', 'CLAUDE.md', 'SETUP.md', 'requirements.txt', 'outputs', 'skills', 'GUIDE.md', '.github']


In [8]:
# Display all actual column names

for i, col in enumerate(df.columns, start=1):
    print(i, repr(col))

1 'content_id'
2 'client_id'
3 'search_volume'
4 'competition'
5 'competition_level'
6 'cpc'
7 'content_type'
8 'main_intent'
9 'word_count'
10 'char_count'
11 'provider_used'
12 'model_used'
13 'impressions_90d'
14 'clicks_90d'
15 'pageviews_90d'
16 'sessions_90d'
17 'users_90d'
18 'engaged_sessions_90d'
19 'ai_sessions_90d'
20 'scroll_events_90d'
21 'days_with_impressions'
22 'days_with_sessions'
23 'impressions_last_30d'
24 'clicks_last_30d'
25 'sessions_last_30d'
26 'impressions_prev_30d'
27 'clicks_prev_30d'
28 'sessions_prev_30d'
29 'content_age_days'
30 'age_tier'
31 'age_tier_order'
32 'days_since_last_update'
33 'freshness_tier'
34 'word_count_tier'
35 'char_count_tier'
36 'ctr'
37 'avg_position'
38 'engagement_rate'
39 'scroll_rate'
40 'ai_traffic_pct'
41 'impression_tier'
42 'position_tier'
43 'trend_direction'
44 'trend_pct'


In [9]:
print("Possible target / label columns:")

for col in df.columns:
    if any(word in col.lower() for word in ["label", "declin", "trend", "flag"]):
        print(col)

Possible target / label columns:
trend_direction
trend_pct


In [10]:
# Inspect possible target column

print("Trend Direction Distribution:")
display(df["trend_direction"].value_counts(dropna=False))

print("\nTrend Percentage Statistics:")
display(df["trend_pct"].describe())

print("\nMissing Values:")
print(df[["trend_direction", "trend_pct"]].isnull().sum())

Trend Direction Distribution:


,count
trend_direction,
down,16262
stable,5962
up,4388
new,2236
flat,1152



Trend Percentage Statistics:


,trend_pct
count,26612.000000
mean,-4.785969
std,473.861780
min,-100.000000
25%,-62.600000
50%,-33.500000
75%,0.000000
max,44900.000000



Missing Values:
trend_direction       0
trend_pct          3388
dtype: int64


In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# SECTION 1: BUILD THE FEATURE VECTOR

import pandas as pd
import numpy as np

# Load dataset
DATA_PATH = "/content/flyrank-ml-workspace2/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_PATH)

print("Dataset Shape:", df.shape)

# Verified numerical features
numeric_candidates = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

# Categorical features
categorical_candidates = [
    "competition_level",
    "content_type",
    "main_intent"
]

numeric_features = [
    col for col in numeric_candidates if col in df.columns
]

categorical_features = [
    col for col in categorical_candidates if col in df.columns
]

print("\nNumerical Features:", numeric_features)
print("\nCategorical Features:", categorical_features)

# Build raw feature dataframe
X_raw = df[numeric_features + categorical_features].copy()

# Convert numerical values
for col in numeric_features:
    X_raw[col] = pd.to_numeric(X_raw[col], errors="coerce")

# avg_position = 0 means unavailable data
if "avg_position" in X_raw.columns:
    X_raw["avg_position"] = X_raw["avg_position"].replace(0, np.nan)

# Create missingness indicators
for col in numeric_features:
    X_raw[f"has_{col}"] = X_raw[col].notna().astype(int)

# Median imputation
for col in numeric_features:
    median_value = X_raw[col].median()
    X_raw[col] = X_raw[col].fillna(
        median_value if pd.notna(median_value) else 0
    )

# Handle categorical missing values
for col in categorical_features:
    X_raw[col] = X_raw[col].fillna("Unknown").astype(str)

# One-hot encoding
X = pd.get_dummies(
    X_raw,
    columns=categorical_features,
    dtype=int
)

print("\nFeature Vector Shape:", X.shape)

print("\nFeature Preview:")
display(X.head())

print("\nMissing Values in Final Features:")
print(X.isnull().sum().sum())

print("\nFeature Columns:")
print(X.columns.tolist())



Dataset Shape: (30000, 44)

Numerical Features: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']

Categorical Features: ['competition_level', 'content_type', 'main_intent']

Feature Vector Shape: (30000, 68)

Feature Preview:


,search_volume,competition,cpc,word_count,char_count,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,...,competition_level_MEDIUM,competition_level_Unknown,content_type_comparison article,content_type_feedly article,content_type_keyword article,main_intent_Unknown,main_intent_commercial,main_intent_informational,main_intent_navigational,main_intent_transactional
0,10.0,0.67,2.05,3221.0,20457.0,3803,29,22,17,16,...,0,0,0,0,1,0,0,0,0,1
1,90.0,0.01,0.05,2481.0,15562.0,15320,7,10,9,9,...,0,0,0,0,1,0,0,1,0,0
2,0.0,0.00,0.00,3515.0,23643.0,12581,11,14,11,11,...,0,0,0,0,1,0,0,1,0,0
3,10.0,0.00,0.00,2877.0,19116.0,11751,58,87,78,75,...,0,0,0,0,1,0,1,0,0,0
4,0.0,0.00,0.00,2803.0,17469.0,19140,24,177,145,144,...,0,0,0,0,1,0,0,1,0,0



Missing Values in Final Features:
0

Feature Columns:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'has_search_volume', 'has_competition', 'has_cpc', 'has_word_count', 'has_char_count', 'has_impressions_90d', 'has_clicks_90d', 'has_pageviews_90d', 'has_sessions_90d', 'has_users_90d', 'has_engaged_sessions_90d', 'has_ai_sessions_90d', 'has_scroll_events_90d', 'has_days_with_impressions', 'has_days_with_sessions', 'has_impressions_last_30d', 'has_clicks_last_30d', 'has_sessions_last_30d', 'has_impressions_prev_30d', 'has_clicks_prev_

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

feature_notes = []

for col in X.columns:
    if col in df.columns:
        dtype = df[col].dtype
        missing_pct = df[col].isnull().mean() * 100
        unique_values = df[col].nunique()

        if pd.api.types.is_numeric_dtype(df[col]):
            feature_type = "Numeric"
            handling = "Median imputation"
        else:
            feature_type = "Categorical"
            handling = "Unknown category / One-hot encoding"

        feature_notes.append({
            "Feature": col,
            "Meaning": "Content-level attribute or performance metric",
            "Type": feature_type,
            "Missing (%)": round(missing_pct, 2),
            "Unique Values": unique_values,
            "Handling": handling,
            "Available at Prediction Time": "Needs verification"
        })

feature_notes_df = pd.DataFrame(feature_notes)

display(feature_notes_df)


,Feature,Meaning,Type,Missing (%),Unique Values,Handling,Available at Prediction Time
0,search_volume,Content-level attribute or performance metric,Numeric,8.23,41,Median imputation,Needs verification
1,competition,Content-level attribute or performance metric,Numeric,8.23,101,Median imputation,Needs verification
2,cpc,Content-level attribute or performance metric,Numeric,8.23,915,Median imputation,Needs verification
3,word_count,Content-level attribute or performance metric,Numeric,25.66,5476,Median imputation,Needs verification
4,char_count,Content-level attribute or performance metric,Numeric,25.66,14839,Median imputation,Needs verification
5,impressions_90d,Content-level attribute or performance metric,Numeric,0.00,9438,Median imputation,Needs verification
6,clicks_90d,Content-level attribute or performance metric,Numeric,0.00,477,Median imputation,Needs verification
7,pageviews_90d,Content-level attribute or performance metric,Numeric,0.00,856,Median imputation,Needs verification
8,sessions_90d,Content-level attribute or performance metric,Numeric,0.00,666,Median imputation,Needs verification
9,users_90d,Content-level attribute or performance metric,Numeric,0.00,644,Median imputation,Needs verification


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
suspicious_columns = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "content_id",
    "client_id"
]

found_suspicious = [
    col for col in suspicious_columns
    if col in df.columns
]

print("Suspicious columns found:")
print(found_suspicious)

print("\nCurrent feature vector check:")

for col in found_suspicious:
    print(f"{col}: {'PRESENT' if col in X.columns else 'NOT PRESENT'}")


Suspicious columns found:
['trend_direction', 'trend_pct', 'content_id', 'client_id']

Current feature vector check:
trend_direction: NOT PRESENT
trend_pct: NOT PRESENT
content_id: NOT PRESENT
client_id: NOT PRESENT


In [17]:
leakage_checks = {
    "trend_direction_in_features": "trend_direction" in X.columns,
    "trend_pct_in_features": "trend_pct" in X.columns,
    "content_id_in_features": "content_id" in X.columns,
    "client_id_in_features": "client_id" in X.columns,
    "label_in_features": "is_declining_label" in X.columns
}

leakage_report = pd.DataFrame(
    list(leakage_checks.items()),
    columns=["Leakage Check", "Detected"]
)

display(leakage_report)

,Leakage Check,Detected
0,trend_direction_in_features,False
1,trend_pct_in_features,False
2,content_id_in_features,False
3,client_id_in_features,False
4,label_in_features,False


In [18]:
potentially_unsafe = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

available_unsafe = [
    col for col in potentially_unsafe
    if col in X.columns
]

print("Features requiring temporal availability review:")
print(available_unsafe)

Features requiring temporal availability review:
['impressions_90d', 'clicks_90d', 'sessions_90d', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

### Leakage Investigation

- `trend_direction` and `trend_pct` are excluded because they are
  directly associated with the declining label.
- `content_id` and `client_id` are identifiers and must not be used
  as predictive features.
- Performance metrics such as impressions, clicks, sessions and CTR
  require temporal validation before model training.
- Features calculated using future observations must be excluded.
- Features that contain information derived from the target must
  never be included in the feature vector.

The presence of a performance metric does not automatically mean
it is leakage. Its availability at the prediction timestamp must
be verified first.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
excluded_features = {
    "content_id": "Unique identifier; not a predictive feature.",
    "client_id": "Client identifier; use for grouping and data splitting.",
    "trend_direction": "Directly related to the declining label.",
    "trend_pct": "Used to derive the declining label; target leakage risk.",
    "is_declining_label": "Target variable; must not be included in X.",
    "provider_used": "Provider metadata; not included in the current feature set.",
    "model_used": "Model metadata; not included in the current feature set.",
    "age_tier": "Categorical representation of content age.",
    "age_tier_order": "Derived ordering of content age.",
    "freshness_tier": "Derived categorical representation of freshness.",
    "word_count_tier": "Derived categorical representation of word count.",
    "char_count_tier": "Derived categorical representation of character count.",
    "impression_tier": "Derived categorical representation of impressions.",
    "position_tier": "Derived categorical representation of position."
}

excluded_df = pd.DataFrame([
    {"Feature": col, "Reason for Exclusion": reason}
    for col, reason in excluded_features.items()
    if col in df.columns
])

display(excluded_df)

,Feature,Reason for Exclusion
0,content_id,Unique identifier; not a predictive feature.
1,client_id,Client identifier; use for grouping and data s...
2,trend_direction,Directly related to the declining label.
3,trend_pct,Used to derive the declining label; target lea...
4,provider_used,Provider metadata; not included in the current...
5,model_used,Model metadata; not included in the current fe...
6,age_tier,Categorical representation of content age.
7,age_tier_order,Derived ordering of content age.
8,freshness_tier,Derived categorical representation of freshness.
9,word_count_tier,Derived categorical representation of word count.


In [20]:
excluded_features = {
    "content_id": "Unique identifier; not a predictive feature.",
    "client_id": "Client identifier; use for grouping and data splitting.",
    "trend_direction": "Directly related to the declining label.",
    "trend_pct": "Used to derive the declining label; target leakage risk.",
    "is_declining_label": "Target variable; must not be included in X.",
    "provider_used": "Provider metadata; not included in the current feature set.",
    "model_used": "Model metadata; not included in the current feature set.",
    "age_tier": "Categorical representation of content age.",
    "age_tier_order": "Derived ordering of content age.",
    "freshness_tier": "Derived categorical representation of freshness.",
    "word_count_tier": "Derived categorical representation of word count.",
    "char_count_tier": "Derived categorical representation of character count.",
    "impression_tier": "Derived categorical representation of impressions.",
    "position_tier": "Derived categorical representation of position."
}

excluded_df = pd.DataFrame([
    {"Feature": col, "Reason for Exclusion": reason}
    for col, reason in excluded_features.items()
    if col in df.columns
])

display(excluded_df)

,Feature,Reason for Exclusion
0,content_id,Unique identifier; not a predictive feature.
1,client_id,Client identifier; use for grouping and data s...
2,trend_direction,Directly related to the declining label.
3,trend_pct,Used to derive the declining label; target lea...
4,provider_used,Provider metadata; not included in the current...
5,model_used,Model metadata; not included in the current fe...
6,age_tier,Categorical representation of content age.
7,age_tier_order,Derived ordering of content age.
8,freshness_tier,Derived categorical representation of freshness.
9,word_count_tier,Derived categorical representation of word count.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.